#Simple generative AI model to discover a book

In [ ]:
# installing needed libraries for web requests, deep learning and transforming
!pip install transformers requests torch

In [ ]:
# library needed for embeddings
!pip install sentence-transformers

In [ ]:
# storing embeddings
!pip install faiss-cpu

In [ ]:
# import needed libraries: request to get the ebook, optional is for code documentation
# SentenceTransformer to transfrom text to vector embeddings, numpy for delaing wiht numbers
# faiss help in finding relevant chunks in the book, pipelin is to use the flant5 model to create answers
import requests
from typing import Optional
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from transformers import pipeline

In [ ]:
# saving the book in a variable
#print("book saved in variable")
url = "Ebook_link"
ebook_text = requests.get(url).text

In [ ]:
# import regular expressions
import re
# making chunks by splitting text on any combination of newlines
chunks = re.split(r'\n\s*\n', ebook_text)
chunks = [chunk.strip() for chunk in chunks if len(chunk.strip()) > 50]
print(f"regex method: {len(chunks)} chunks")

regex method: 780 chunks


In [ ]:
# creating embeddings
#print("creating embeddings")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(chunks, show_progress_bar=True)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
# create FAISS index for similarity search
#print("building search index")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

In [ ]:
# use Flan-T5
# free generative model that creates natural responses
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512,
    truncation=True
)

Device set to use cpu


In [ ]:
# create generative chatbot function
def ask_about_book(question, top_k=3):
    """
    ask questions about the book and get AI-powered repsonses
    Args:
       question: your question about the book
       top_k: How many relevant passages to search (default 3)
    """
    # find most relevant parts
    question_embedding = embedding_model.encode([question])
    distances, indices = index.search(np.array(question_embedding).astype('float32'), top_k)
    # combine the relevant passages
    context = "\n\n".join([chunks[idx] for idx in indices[0]])
    # limit context length to not overwhelm the model
    if len(context) > 1000:
        context = context[:1000] + "..."
    # create a prompt for the model
    prompt = f"""based on the following passages from pride and prejudice, answer question in a conversational way.
context:{context}
Question: {question}
Answer:"""
    try:
        # Generate the answer!
        result = generator(prompt, max_length=200, min_length=20, do_sample=True, temperature=0.7)
        answer = result[0]['generated_text']
        print(f"\n AI generated answer:")
        print(f"{answer}")
        print(f"\n based on these passages and parts from the books:")
        print(context[:400] + "..." if len(context) > 400 else context)

        return answer
    except Exception as e:
        print(f"Error occurred: {e}")
        print(f"\n here are the relevant passages i found:")
        print(context[:800])
        return "Sorry, i encounterd an issue generating the response"

In [ ]:
# test it
print("Generative AI ebook chatbot ready!")
print("-"*20)
print("This AI will generate natural responses based on the book content")

Generative AI ebook chatbot ready!
--------------------
This AI will generate natural responses based on the book content.


In [ ]:
# example questions
questions = ["who is the writer?",
             "where was the book published?",
             "what is the hero first name?",
             "describe the book"]
for q in questions:
    print(f"\n question:{q}")
    ask_about_book(q)
    print("\n" + "-"*10)


 question:who is the writer?

 AI generated answer:
Judith Boss, Christy Phillips, Lynn Hanninen and David Meltzer

 based on these passages and parts from the books:
<p><strong>Credits</strong>: Judith Boss, Christy Phillips, Lynn Hanninen and David Meltzer. HTML version by Al Haines.<br>
        Further corrections by Menno de Leeuw.</p>

<p>
“Justine Moritz! Poor, poor girl, is she the accused? But it is wrongfully;
every one knows that; no one believes it, surely, Ernest?”
</p>

<p>
Our conversations are not always confined to his own history and misfo...

----------

 question:where was the book published?

 AI generated answer:
a class="reference external" href="https://www.gutenberg.org">www.gutenberg.org/a>

 based on these passages and parts from the books:
<div>This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of 

In [ ]:
# Interactive mode
print("\n\n" + "-"*10)
print("Interactive mode - ask questions")
print("-"*10)
print("Type your questions below (type 'quit to exit)\n")
while True:
    user_question = input("the question: ")
    if user_question.lower() in ['quit', 'exit', 'q']:
        print("\n thanks for the chat, goodbye.")
        break
    if user_question.strip():
        ask_about_book(user_question)
        print()



----------
Interactive mode - ask questions
----------
Type your questions below (type 'quit to exit)

the question: in what year the book was published?

 AI generated answer:
1908/i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>//i>

 based on these passages and parts from the books:
<p>
“I can hardly describe to you the effect of these books. They produced in me an
infinity of new images and feelings, that sometimes raised me to ecstasy, but
more frequently sunk me into the lowest dejection. In the <i>Sorrows of
Werter</i>, besides the interest of its simple and affecting story, so many
opinions are canvassed and so many lights thrown upon what had hitherto been to
me o...

the question: quit

 thanks for the chat, goodbye.
